# Homework 02: SQL Mini-Project with Sakila
## Joins, Aggregation, CTEs, Window Functions, and Performance

**Due date:** _April 3, 2026 by 11:59 PM_

**Points:** 100 total

> **Name (FIRST and LAST)** — Jonathan Le Roux

- - -

## Overview and Instructions

**Context:** This homework builds directly on our relational database + SQL module (Days 11–19). You will use a Dockerized MySQL database (the Sakila sample database you set up previously) and a Jupyter notebook to carry out a small, portfolio-ready SQL analysis.

You will:
- Connect to a Dockerized MySQL database from this notebook.
- Design and implement several non-trivial SQL queries using Sakila.
- Use joins, grouped aggregation with `HAVING`, CTEs, window functions, and a brief performance check.
- Practice explicit validation habits (row counts, sanity checks, cross-checks).
- Produce at least one manager-style report table that could stand alone in a portfolio.
- Reflect on your SQL reasoning and debugging process.
- Log any use of AI tools in an **AI Audit Log** section.
- Create a GitHub repository to share your work.

**If you get stuck, have questions, or run into any issues as you work through this assignment**, remember to consult the PCAs, ICAs, slides from class for the **Day 11-19 class periods** -- the slides in particular often have worked examples of queries that are similar to what you'll need to write for this assignment. You are also encouraged to **come to office hours** to get direct support from your instructors! If you can't find out office hours information or need to schedule by appoint, **send us an email or catch one of us at the end of class**.

**Submission:**
- Push this completed notebook (and any small helper files) to a new private GitHub repo called `cmse492-hw02-yourMSUNetID` (make sure you use your actual MSU NetID!). **Details for this process are include in Part 10 below.**
- Submit the GitHub URL via the provided Microsoft form.
- Upload the executed notebook (with outputs) to D2L as a backup.

This notebook is designed to be a **self-contained artifact**: someone with access to your Docker image and credentials should be able to rerun your analysis and understand what you did, why you did it, and how you validated it.

**Grading breakdown:** See section headings for point values.

<div class="alert alert-attention">

**Note about AI tools:** As discussed at the beginning of the course, you are allowed to use AI tools (e.g., ChatGPT, Copilot) in responsible, transparent, and ethical ways. For this particular assignment, if you end up using AI tools for assignment, but you will be expected to document your usage in an "AI Audit Log" section near the end of this notebook. See the details in the "AI Audit Log" section below.

</div>


- - -

## 0. Setup and Database Connection

In this section you will:
- Confirm your Dockerized MySQL + Sakila setup is working.
- Establish a connection from Python to the database using `sqlalchemy`.
- Define a small helper function to run SQL and get results as a `pandas` DataFrame (make sure you look at this function so that you understand how it works and can use it in the assignment!)

**Even though you are being provided with the code cell necessary to connect to the database, you should make sure you carefully review the code and understand how it is working.** If you end up needing to connect to a SQL database on your own in the future, you should know how to do so.

**Remember**: Before you try to connect to the database, make sure your Docker container is running and that the Sakila database is properly set up inside it. If you don't recall how to do this, review the instructions from the Day 17 content and get that set up first.

If you run into an issue with this part of the assignment, contact your instructors _as soon as possible_ so we can help you get it resolved. You will not be able to complete the rest of the assignment without a working database connection, so this is a critical first step.


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Credentials for the local Docker MySQL container (must match docker-compose.yml)
uname = 'sakila'
pwd = 'p_ssW0rd'
hname = 'localhost'
dbname = 'sakila'

engine = create_engine(f'mysql+pymysql://{uname}:{pwd}@{hname}/{dbname}')
connection = engine.connect()

def run_sql(query, params=None):
    """Run a SQL query and return a pandas DataFrame.
    `query` can be a string; `params` is an optional dict for parameterized queries.
    """
    with engine.connect() as conn:
        result = conn.execute(text(query), params or {})
        df = pd.DataFrame(result.fetchall(), columns=result.keys())
    return df

# Quick test to confirm that you can connect and query the database
test_df = run_sql("SELECT * FROM film LIMIT 5;")
test_df.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,1.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2026-03-09 13:51:57
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


- - -

## 1. Framing Your Mini-Project (5 points)

Your goal is to design a **small analytic story** using the Sakila database. Think of a manager at a video rental company who wants to understand something about customers, films, inventory, stores, or revenue.

In this section, you will define:
- A brief business/analytic question or set of questions you want to answer using SQL and the audience you are imagining.
- The main tables you expect to use.
- The kinds of outputs you want (e.g., a ranked table, grouped summary, percent-of-total view).

**Note**: you might find that your question and plans evolve as you start writing SQL and seeing results. This is totally normal! Just make sure to document your initial thinking here, and then as you evolve your question and plans, make sure to update this section to reflect that evolution. Make sure to keep your initial question and plans in there as well, so that we can see how your thinking evolved.

**Another note**: You might need to spend some time exploring the Sakila schema and data to come up with a question that is interesting and feasible. You can access the DB using Adminer (via `http://localhost:8080`) to explore the schema and/or use SQL queries to poke at the data as part of your process of coming up with a good question.

If you're really struggling to come up with your "story", you can start working through the SQL exercises in the sections that follow, which require you to come up with individual questions to answer with SQL. **You might find that as you do those exercises, you see a collection of insights that your can develop into your project story.** "Reverse-engineering" your project story from the SQL exercises is a totally valid way to come up with a project question and plan!

### 1.1 Project question and plan

**Prompt:** In 1–2 paragraphs, describe your mini-project:
- What is the main question or set of questions you want to answer using SQL?
- Who is the imagined audience (e.g., store manager, marketing analyst, operations lead)?
- Which core tables do you expect to rely on (list at least 3 Sakila tables and why)?
- What kind of final table(s) or figure(s) do you expect to produce (e.g., top categories by revenue, customer segments by activity)?




Potential questions:
- What age rating is most profitable?
- Are there some movies that should not be kept on the shelves because they are never rented
- What movies should be kept at the front of the store to bring in window shoppers
- are longer movies more enjoyable and more likely to be rented
- what language of movies are preferred


The imagined audience is a store owner who wants to increase profit and the inventory manager who might want to know what people are renting so that those things could be used for advertising

film - it has rental rates and names
<br>film_category - it links films and categories
<br>inventory - it shows the film the last time it was updated

final outcomes:
- movies to sell and remove from rental pool
- top ratings by revenue
- most profitable movies
- most rented films by rating
- how many of each movie we have
>

---

## 2. Joins and Multi-Table Queries (15 points)

In this section, you will write **at least two non-trivial join queries** that connect 2–3 tables with meaningful conditions (not just a single key lookup). For each query:
- Clearly state the question it answers.
- Sketch what a single row in the result represents.
- Write and run the SQL.
- Perform at least one validation check (row counts, spot-check rows against base tables, etc.).


### 2.1 Join Query 1

**Example idea (you may adapt this or pick a different one, do not use exactly this example):**
- "For each rental in a specified 60-day period, show customer name, film title, rental date, and store city."

Based on your question and plans from the previous section, write your own question and design for this first join query. You can use the example above as a template or inspiration, but make sure to write your own unique question and design that fits with your overall mini-project. For this first join query, you're expected to clearly define the following:

1. **Question in words**: Describe what you are trying to learn in one or two sentences.
2. **Row meaning sketch**: What does each row represent? Which columns should appear?
3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?
4. **SQL query**: Write the join with clear aliases.
5. **Validation**: Perform at least one validation check, such as:
   - Compare a row count to a simpler query.
   - Spot-check an individual record across base tables.


> **2.1.a Question and design**
> 
**IDEA:** Give the film name and the store address of the films that haven't been rented in the longest time

1. **Question in words**: Which films have not been rented for the longest time, and at what stores are those films not being rented?

2. **Row meaning sketch**: film_name, store_adress, last_time_rented

3. **Tables and keys**: Which tables are you joining (e.g., `rental`, `customer`, `inventory`, `film`, `store`, `address`, `city`)? On which key columns?

joining: 
- film & inventory on film_id
- inventory & store on store_id
- inventory & rental on inventory_id
- store & address on address_id

In [38]:
# 2.1.b Join Query 1 - SQL
join_query_1 = """
SELECT film.title, address.address AS store_address,MAX(rental.rental_date) AS last_rented
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN store ON store.store_id = inventory.store_id
JOIN address ON address.address_id = store.address_id
JOIN rental ON rental.inventory_id = inventory.inventory_id
GROUP BY film.title, address.address
ORDER BY last_rented, film.title;
"""

df_join_1 = run_sql(join_query_1)
df_join_1.head()

,title,store_address,last_rented
0,GRACELAND DYNAMITE,47 MySakila Drive,2005-08-17 07:46:54
1,DIRTY ACE,28 MySQL Boulevard,2005-08-17 11:42:45
2,PERSONAL LADYBUGS,28 MySQL Boulevard,2005-08-17 12:44:27
3,UNFORGIVEN ZOOLANDER,28 MySQL Boulevard,2005-08-17 16:10:19
4,PAPI NECKLACE,28 MySQL Boulevard,2005-08-17 17:16:42


**2.1.c Validation for Join Query 1**

Describe at least one validation check you ran and what you concluded based on it.

You might:
- Show a supporting SQL snippet in a small code cell (e.g., `COUNT(*)` comparison).
- Spot-check one ID across multiple tables and describe what you saw.


In [35]:
# Example based on the sample query above:
# check total rentals in the same date range without joins

validation_query_1 = """
SELECT film.title, address.address AS store_address,rental.rental_date AS rental_dates
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN store ON store.store_id = inventory.store_id
JOIN address ON address.address_id = store.address_id
JOIN rental ON rental.inventory_id = inventory.inventory_id
WHERE title = 'GRACELAND DYNAMITE'
ORDER BY rental.rental_date DESC
"""
run_sql(validation_query_1)

,title,store_address,rental_dates
0,GRACELAND DYNAMITE,47 MySakila Drive,2005-08-17 07:46:54
1,GRACELAND DYNAMITE,47 MySakila Drive,2005-08-17 04:30:09
2,GRACELAND DYNAMITE,47 MySakila Drive,2005-08-01 12:44:17
3,GRACELAND DYNAMITE,47 MySakila Drive,2005-07-31 11:29:23
4,GRACELAND DYNAMITE,47 MySakila Drive,2005-07-10 19:54:41
5,GRACELAND DYNAMITE,47 MySakila Drive,2005-07-10 14:10:22


In [32]:
# Put your own validation query(ies) here and explanation in the next section.
validation_query_2 = """
SELECT film.title, address.address AS store_address,rental.rental_date AS rental_dates
FROM film
JOIN inventory ON film.film_id = inventory.film_id
JOIN store ON store.store_id = inventory.store_id
JOIN address ON address.address_id = store.address_id
JOIN rental ON rental.inventory_id = inventory.inventory_id
WHERE title= 'DIRTY ACE'
ORDER BY address.address, rental.rental_date
"""
df_validation = run_sql(validation_query_2)
df_validation

,title,store_address,rental_dates
0,DIRTY ACE,28 MySQL Boulevard,2005-05-26 13:40:40
1,DIRTY ACE,28 MySQL Boulevard,2005-06-15 12:32:13
2,DIRTY ACE,28 MySQL Boulevard,2005-07-09 03:33:32
3,DIRTY ACE,28 MySQL Boulevard,2005-08-02 13:32:00
4,DIRTY ACE,28 MySQL Boulevard,2005-08-02 20:21:08
5,DIRTY ACE,28 MySQL Boulevard,2005-08-17 09:03:31
6,DIRTY ACE,28 MySQL Boulevard,2005-08-17 11:42:45
7,DIRTY ACE,47 MySakila Drive,2005-06-16 09:42:48
8,DIRTY ACE,47 MySakila Drive,2005-07-10 20:51:34
9,DIRTY ACE,47 MySakila Drive,2005-07-30 01:29:48


> **_Write your validation explanation here._**

validation query 1: I spot checked the record for one of the titles to see all of the rental_dates that the movie has been rented for. Then I ensure that the most recent rental date was the same as entry in the original query. They were the same.

validation query 2: I wanted to ensure that the ordering of the last rental was correct even based on the store. So I spot checked a record that was in both stores and ensured that the most recent at the correct store was being reflected in the intitial query.
 

### 2.2 Join Query 2

Create a **different** join that still aligns with your project theme.

**Example ideas (again, you should adapt these to your project or create your own):**
- Customer-level summary: each row is one customer with their total rentals, number of distinct categories they rent, and home city.
- Inventory-level view: each row is one film copy with film title, store, and how many times it has been rented.

Repeat the pattern as before, clearly defining:
1. Question in words.
2. Row meaning sketch.
3. Tables and keys.
4. SQL query.
5. Validation check(s).

> **2.2.a Question and design**
> 
1. What film rating is the most common in the stores?
2. film rating | total_copies. This shows each type of film rating and the number of films we have of those ratings
3. film & inventory are the tables being used. They are joined on the film_id key

In [14]:
# 2.2.b Join Query 2 - SQL
join_query_2 = """
SELECT film.rating, COUNT(*) AS total_copies
FROM film
INNER JOIN inventory ON film.film_id = inventory.film_id
GROUP BY film.rating
ORDER BY total_copies DESC;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_join_2 = run_sql(join_query_2)
df_join_2.head()

,rating,total_copies
0,PG-13,1018
1,NC-17,944
2,PG,924
3,R,904
4,G,791


**2.2.c Validation for Join Query 2**

As before, describe your validation steps and conclusions for this second join query. You can use similar validation techniques as before or come up with new ones that fit the specific question and data you are working with.

In [39]:
# As appropriate, define helper validation query(ies) for Join Query 2
validation_query_2 = """
SELECT COUNT(*)
FROM inventory 
"""
run_sql(validation_query_2)


,COUNT(*)
0,4581


In [40]:
print(df_join_2['total_copies'].sum())

4581


> **_Describe your validation steps and conclusions here._**

validation_query_2 : I wanted to ensure that the total number of films in the inventory corresponds with the total of the number of films by category. I used python to calculate the total number of films based off of the intitial query and then I used a select query to check that value based on the table.

---

## 3. Grouped Aggregation with HAVING (15 points)

In this section you will:
- Write at least one grouped aggregation that uses `GROUP BY` and `HAVING`.
- Apply sanity checks (e.g., sum of group counts vs. raw counts, spot-check of a group).
- Tie the result to a question your audience might actually care about.


### 3.1 Aggregation Query with HAVING (required)

**Example ideas (adapt or design your own to fit your audience and align with your mini-project):**
- "Find film categories with at least 1000 total rentals, ordered from most to least rented."
- "Find customers who have spent at least \$X in total payments."

For your query, **document**:
1. Business question in words (1–2 sentences).
2. Row meaning and expected columns in the grouped result.
3. SQL query with `GROUP BY` and `HAVING`.
4. At least two sanity checks (e.g., compare grouped count sums to raw counts, spot-check a single group).


> **3.1.a Question and design**
> 
1. Which films have more than 5 copies and also have the highest average rental rate - this would be a proxy for what movies to put close to the entrance of the store.
2. Each row will have the title of the movie, rental rate of that movie and the total number of copies of that movie we have

In [49]:
# 3.1.b Aggregation with GROUP BY + HAVING

agg_query = """
SELECT film.title, AVG(film.rental_rate) AS average_rental_rate, COUNT(inventory.inventory_id) AS total_copies
FROM film
INNER JOIN inventory ON film.film_id = inventory.film_id
GROUP BY film.film_id
HAVING COUNT(inventory.inventory_id) > 5 and average_rental_rate = (SELECT MAX(film.rental_rate) FROM film)
ORDER BY total_copies DESC, film.rental_rate,film.title
"""

# Once you've defined your query, you can uncomment/edit the lines below to run it and see the results.
df_agg = run_sql(agg_query)
display(df_agg)

,title,average_rental_rate,total_copies
0,APACHE DIVINE,4.990000,8
1,BOOGIE AMELIE,4.990000,8
2,BUCKET BROTHERHOOD,4.990000,8
3,CAT CONEHEADS,4.990000,8
4,CONFIDENTIAL INTERVIEW,4.990000,8
...,...,...,...
114,TRUMAN CRAZY,4.990000,6
115,VELVET TERMINATOR,4.990000,6
116,WHISPERER GIANT,4.990000,6
117,WORKING MICROCOSMOS,4.990000,6


**3.1.c Sanity checks for aggregation**

Describe at least **two** checks you performed. Ideas:
- Compare the sum of group counts to a raw `COUNT(*)` over a comparable subset.
- Focus on one group (e.g., one category or one customer), query its underlying rows, and verify the count.
- If applicable, use a `LEFT JOIN` to see categories with zero values and explain what you observe.


In [53]:
# As appropriate, write helper sanity-check queries for aggregation here and
# provide explanation in the next section.

check_query_1 = """
SELECT film.title, COUNT(film.title) AS total_copies
FROM film
INNER JOIN inventory ON film.film_id = inventory.film_id
WHERE film.title= 'APACHE DIVINE'
"""
df_val1 = run_sql(check_query_1)

check_query_2 = """
SELECT film.title, AVG(film.rental_rate) as avg_rental_rate
FROM film
INNER JOIN inventory ON film.film_id = inventory.film_id
WHERE film.title= 'APACHE DIVINE'
"""

df_val2 = run_sql(check_query_2)

check_query_3 = """
SELECT MAX(film.rental_rate) as max_rental_rate
FROM film
"""
df_val3= run_sql(check_query_3)

display(df_val1)
display(df_val2)
display(df_val3)

,title,total_copies
0,APACHE DIVINE,8


,title,avg_rental_rate
0,APACHE DIVINE,4.990000


,max_rental_rate
0,4.99


> **_Write your sanity-check explanation here._**

check query 1 - Checks the corresponding number of copies for a specific movie. This number correlates with the number that is stated in the original query for the movie.
check query 2 & 3 - Ensures that the average_rental rate is being calculated correctly and it corresponds with the max_rental_rate

- - - 

## 4. Multi-Step Query Using a CTE (15 points)

Now you will use a **CTE (`WITH` clause)** to break a complex analysis into readable steps. You may reuse logic from earlier sections, but the CTE should add structure (e.g., pre-filtering, intermediate grouping) rather than just rename an existing query.

**Example patterns:**
- First CTE: compute customer-level rental counts and total payments.
- Second step: filter to active customers, compute ranks or thresholds on top.

Your CTE query must:
- Use at least one `WITH` clause.
- Do something non-trivial in the CTE (filtering, joining, grouping, etc.).
- Be accompanied by validation (e.g., run parts of the CTE separately, check a subset).

### 4.1 CTE design and explanation

Describe your CTE in words:
- What does the CTE compute?
- What does the outer query do on top of it?
- How does this decomposition make the logic clearer than a single big query would be?


> **_Write your CTE design explanation here._**
1. The CTE computes the total revenue for each film. It then returns the film_id, its rating and the total revenue of that film
2. The outer query groups the results of the CTE based on the rating and returns the average revenue based on the rating
3. It is easier to calculate the total revenue of each film first in a CTE and then find the average of the revenues otherwise you would have to use a combination of SUM and AVG in a more complicated syntax. This would make it harder to debug.


### 4.2 Writing the CTE query

Now that you have a plan for your CTE, write the SQL query that implements it.

In [60]:
# 4.2 CTE-based query

cte_query = """
WITH film_revenues AS (
    SELECT film.film_id, film.rating, SUM(payment.amount) AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id, film.rating
)
SELECT rating,AVG(film_revenue) AS avg_revenue
FROM film_revenues
GROUP BY rating
ORDER BY avg_revenue DESC
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_cte = run_sql(cte_query)
display(df_cte)

,rating,avg_revenue
0,PG,72.884754
1,PG-13,71.639249
2,R,70.212646
3,NC-17,68.688465
4,G,68.211871


### 4.3 CTE validation

Run at least one validation step focused on the **intermediate CTE logic**, such as:
- Select a subset of the CTE output (e.g., `LIMIT 10`) and manually compare to base tables.
- Recompute a simpler version of a metric (e.g., total payments) for one customer and confirm it matches the CTE result.

In [64]:
# As appropriate, write helper queries to inspect the CTE result or base tables
cte_part = """
SELECT film.film_id, film.rating, SUM(payment.amount) AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id, film.rating
    LIMIT 1
"""
print(run_sql(cte_part))

check_cte_row = """
SELECT film.film_id,payment.amount AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    WHERE film.film_id=1
"""
valid_df = run_sql(check_cte_row)
sum(valid_df['film_revenue'])

   film_id rating film_revenue
0        1     PG        36.77


Decimal('36.77')

> **_Describe your validation and what you concluded here. Include helper queries in code cells if needed._**

I checked the CTE part of the query by limiting it to the first row so that i could see the film_id and the film_revenue.
I then made a validation query that gets all the entries where the film made revenue and then I used python to sum up those 
revenues to ensure that the values correlated. The values correlated which shows that the CTE part of the query worked correctly.

---

## 5. Window Function with `OVER (...)` (15 points)

Next, you will write at least one query that uses a **window function**. Options include:
- `ROW_NUMBER()`, `RANK()` or `DENSE_RANK()` within a category (e.g., top customers per store).
- A percent-of-total calculation using `SUM(...) OVER ()` or `SUM(...) OVER (PARTITION BY ...)`.

Your window query should:
- Use `OVER (...)` in a meaningful way.
- Produce a result that could help your audience understand ranking or relative importance.
- Include at least one validation step (e.g., verify that percentages sum to ~100%).


### 5.1 Window function design

Describe your plan:
- What are you ranking or computing percentages over?
- What does each row represent in the final result?
- How will your audience interpret the window column(s)?


> **_Write your window function plan here._**

1. I am ranking the most profitable films based on the amount of revenue that they earn above the average earnings of a film in their same rating.
2. Each row represents the most profitable titles in each of the age ratings.
3. The user will be able to see the ranking of how profitable each film is. i.e. Where does this film stand compared to all others in terms of profitability? This will also show what films are doing the worst in terms of contributing to the bottom line

### 5.2 Writing the window function query

Now that you have a plan for your window function, write the SQL query that implements it.

In [93]:
# 5.2 Window function query

window_query = """
WITH film_revenues AS (
    SELECT film.film_id, film.title, film.rating, SUM(payment.amount) AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id, film.rating
),
rating_avg_revenue AS (
    SELECT rating,AVG(film_revenue) AS avg_revenue
    FROM film_revenues
    GROUP BY rating
)
SELECT
    film_revenues.title,
    film_revenues.rating,
    film_revenues.film_revenue,
    rating_avg_revenue.avg_revenue AS avg_revenue_of_rating,
    film_revenues.film_revenue - rating_avg_revenue.avg_revenue AS revenue_above_avg,
    RANK() OVER (
        ORDER BY film_revenues.film_revenue - rating_avg_revenue.avg_revenue DESC
    ) AS overall_profit_rank
FROM film_revenues
INNER JOIN rating_avg_revenue ON rating_avg_revenue.rating = film_revenues.rating
ORDER BY overall_profit_rank, film_revenues.title
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_window = run_sql(window_query)
df_window.head()

,title,rating,film_revenue,avg_revenue_of_rating,revenue_above_avg,overall_profit_rank
0,TELEGRAPH VOYAGE,PG,231.73,72.884754,158.845246,1
1,WIFE TURN,NC-17,223.69,68.688465,155.001535,2
2,ZORRO ARK,NC-17,214.69,68.688465,146.001535,3
3,GOODFELLAS SALUTE,PG,209.69,72.884754,136.805246,4
4,SATURDAY LAMBS,G,204.72,68.211871,136.508129,5


### 5.3 Window function validation

Identify at least one validation you can perform, such as:
- Summing `pct_of_total` and confirming it is close to 100 (allowing for rounding).
- Checking that the ranking order matches the sorted totals.
- Comparing one row’s percent-of-total to a hand calculation.

In [85]:
# As appropriate, write helper queries or pandas checks for window function results

ranking_query_validation = """
SELECT COUNT(DISTINCT film_id) as number_of_films
FROM inventory
"""

if df_window['overall_profit_rank'][len(df_window['overall_profit_rank'])-1] <= run_sql(ranking_query_validation)['number_of_films'][0]:
    '''
    the reason we use <= is because if the revenue_above the average is the same for 2 films which in this case it is not then
    the last rank would be lower than the total number of films in the database.
    '''
    print(True)
else:    
    print(False)

True


> **_Write your validation explanation here and include helper code if needed._**

The ranking_query_validation query returns the total number of distinct films that there are in the inventory.
The python code checks that the last entry in the window function has the same rank or a lower rank than the total number of distinct films. This ensures that the ranking worked properly. If there was a rank higher then we know that something went wrong with the ranking. Secondly you can visually observe that the ranking correlates with the revenue_above_avg column which shows how much above the average each film is.

---

## 6. Performance Check with `EXPLAIN` and/or Index (5 points)

Pick **one** of your more complex queries from the earlier sections (join, aggregation, or window-based) and perform a brief performance analysis using MySQL's `EXPLAIN` statement.

You should:
- State which query you are examining and what aspect of its execution plan you want to inspect.
- Run `EXPLAIN` on the query and capture the output in a DataFrame.
- Identify at least one aspect of the plan (e.g., table scan vs. index usage, join order, estimated rows) and explain why it seems reasonable or concerning.
- Optionally, propose or implement a simple index and compare the `EXPLAIN` output before vs. after.

You do **not** need to time queries precisely or chase microseconds. The focus is on reading and interpreting the query plan.

**Note**: You might not see any "red flags" in the `EXPLAIN` output, especially on a small dataset like Sakila. That's totally fine! The point is to practice reading the plan and thinking about what it means.

### 6.1 Choose a query and plan your analysis

Before running `EXPLAIN`, take a moment to describe your plan:
1. Which query from an earlier section are you examining (e.g., "Section 3 aggregation by category", "Section 5 window query")?
2. Why did you choose this query (e.g., it joins many tables, it aggregates a large dataset)?
3. What do you expect to see in the `EXPLAIN` output (e.g., index usage on join keys, a full scan on a particular table)?

> **_Write your plan here._**
>
1. I am going to perform the explain on my window query since it is built off of of my CTE query and it is the hardest query that I have written.
2. It implements CTEs, joins CTE's and performs a window function
3. I don't really know what to expect. The query is complex but I think the number of rows should be minimal since I made the CTE's to increase efficiency instead of using indented selects. I expect to see a table for each of the CTE's

### 6.2 Run EXPLAIN on your chosen query

Now that you have a plan, run `EXPLAIN` on the query you chose. Note that you can prefix any SQL query with `EXPLAIN` to get the execution plan — `run_sql()` works with `EXPLAIN` statements just like any other query.

In [87]:
# 6.2 Run EXPLAIN on your chosen query

# Prefix your query with EXPLAIN to capture the execution plan.
# Note: run_sql() works with EXPLAIN statements just like any other query.

explain_query = """
EXPLAIN WITH film_revenues AS (
    SELECT film.film_id, film.title, film.rating, SUM(payment.amount) AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id, film.rating
),
rating_avg_revenue AS (
    SELECT rating,AVG(film_revenue) AS avg_revenue
    FROM film_revenues
    GROUP BY rating
)
SELECT
    film_revenues.title,
    film_revenues.rating,
    film_revenues.film_revenue,
    rating_avg_revenue.avg_revenue AS avg_revenue_of_rating,
    film_revenues.film_revenue - rating_avg_revenue.avg_revenue AS revenue_above_avg,
    RANK() OVER (
        ORDER BY film_revenues.film_revenue - rating_avg_revenue.avg_revenue DESC
    ) AS overall_profit_rank
FROM film_revenues
INNER JOIN rating_avg_revenue ON rating_avg_revenue.rating = film_revenues.rating
ORDER BY overall_profit_rank, film_revenues.title
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_explain = run_sql(explain_query)
display(df_explain)

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,PRIMARY,<derived3>,None,ALL,NaN,NaN,NaN,NaN,2,100.0,Using where; Using temporary; Using filesort
1,1,PRIMARY,<derived2>,None,ref,<auto_key0>,<auto_key0>,2,rating_avg_revenue.rating,2,100.0,NaN
2,3,DERIVED,<derived2>,None,ALL,NaN,NaN,NaN,NaN,2,100.0,Using temporary
3,2,DERIVED,rental,None,index,"PRIMARY,idx_fk_inventory_id",idx_fk_inventory_id,3,NaN,1,100.0,Using index; Using temporary
4,2,DERIVED,payment,None,ref,fk_payment_rental,fk_payment_rental,5,sakila.rental.rental_id,1,100.0,NaN
5,2,DERIVED,inventory,None,eq_ref,"PRIMARY,idx_fk_film_id",PRIMARY,3,sakila.rental.inventory_id,1,100.0,NaN
6,2,DERIVED,film,None,eq_ref,PRIMARY,PRIMARY,2,sakila.inventory.film_id,1,100.0,NaN


### 6.3 Interpret the EXPLAIN output

Now that you have the `EXPLAIN` output, interpret it in a few sentences. Address at least one of the following:
- Is MySQL using an index or doing a full table scan on any of the tables? Does this seem reasonable given the query structure?
- What is the estimated number of rows being scanned for each table (the `rows` column)? Does this seem appropriate?
- Is the join order what you would expect? Why or why not?
- Based on the plan, would you suggest any changes to improve performance (e.g., adding an index, restructuring the query)?

> **_Write your performance interpretation here (2–5 sentences)._**

1. It is only doing a full table on two of the derived tabels that are created by the CTE's. But this should not matter since the number of rows in those tables are low. It is using a key on 4 of the other tables. Inventory and film are using their primary keys and rental and payment are using indexes
2. The derived tables each have 2 rows and the other tables all have 1 row. This seems low but appropriate because most of the joins are using indexes.
3. The join order is what expected because the query first needs to make the CTE tables before they can be joined onto the main tables.
4. I would not make any changes since the total number of work being done is low. Even though some of the tables scan all the records the number of rows is low enough for it not to matter


### 6.4 Optional: Propose or add an index

If you'd like to go further and it seems like an optimization is needed, you can propose or create a simple index and compare the `EXPLAIN` output before vs. after. Document:
- Which column(s) you chose and why (e.g., frequently used in `WHERE` or join conditions).
- Whether the `EXPLAIN` output changed in a way that suggests better performance (e.g., less full-table scanning, lower estimated rows).

I think that my query is efficient enough. I say this because the number of rows in each table is low so even if you mulitply the number of rows in each table together then it is still a low number of rows and a low number of work being done

---

## 7. Manager-Style Report Table (10 points)

Design **one table** that you would feel comfortable showing to your imagined audience or including in a portfolio. This table should:
- Be relatively **wide** (e.g., 4–8 columns) with clear, human-readable column labels.
- Summarize something decision-relevant (e.g., categories by revenue and share, top customers per store).
- Optionally use `CASE` expressions or window functions to create computed or pivot-like columns.

You may reuse or adapt one of your earlier queries if it already reads like a manager report, or you may write a dedicated query here.

### 7.1 Describe your report

Before writing the SQL, describe your report:
1. Who is the audience for this table (e.g., store manager, marketing analyst, operations lead)?
2. What decision or question does this table support?
3. Which columns will appear, and what does each one tell the audience?
4. Will you reuse a query from an earlier section or write a new one?

> **_Write your audience, decision, and column plan here._**
1. Store manager/ person who decides what films to keep in the store
2. What is the most profitable film and how many of them do we have.
3. 
- title	-the title of the movie
- total_copies	- how many copies of the movie the business owns
- rating - the age rating of the movie
- film_revenue	- how much money the film has earned
- revenue_above_avg	- how much above the average revenue of that age rating type of film does this film make
- overall_profit_rank - the rank that indicates profitability based off of how much money above the avg_revenue the film makes
4. I combined two different queries.
>

### 7.2 Write the report query

Now write the SQL query that produces your report table. Aim for clear, human-readable column aliases and a layout that could stand on its own in a portfolio or presentation.

In [110]:
# 7.2 Report query SQL

report_query = """
WITH film_revenues AS (
    SELECT film.film_id, film.title, film.rating, SUM(payment.amount) AS film_revenue
    FROM film
    INNER JOIN inventory ON film.film_id = inventory.film_id
    INNER JOIN rental ON inventory.inventory_id = rental.inventory_id
    INNER JOIN payment ON rental.rental_id = payment.rental_id
    GROUP BY film.film_id, film.rating
),
rating_avg_revenue AS (
    SELECT rating,AVG(film_revenue) AS avg_revenue
    FROM film_revenues
    GROUP BY rating
),
film_number_of_copies AS (
SELECT film.film_id, film.title,  COUNT(inventory.inventory_id) AS total_copies
FROM film
INNER JOIN inventory ON film.film_id = inventory.film_id
GROUP BY film.film_id, film.title
)
SELECT
    film_revenues.title,
    film_number_of_copies.total_copies,
    film_revenues.rating,
    film_revenues.film_revenue,
    film_revenues.film_revenue - rating_avg_revenue.avg_revenue AS revenue_above_avg,
    RANK() OVER (
        ORDER BY film_revenues.film_revenue - rating_avg_revenue.avg_revenue DESC
    ) AS overall_profit_rank
FROM film_revenues
INNER JOIN rating_avg_revenue  ON rating_avg_revenue.rating = film_revenues.rating
INNER JOIN film_number_of_copies ON film_number_of_copies.film_id = film_revenues.film_id
ORDER BY overall_profit_rank;
"""

# Once you've defined your query, you can uncomment the lines below to run it and see the results.
df_report = run_sql(report_query)
df_report

,title,total_copies,rating,film_revenue,revenue_above_avg,overall_profit_rank
0,TELEGRAPH VOYAGE,7,PG,231.73,158.845246,1
1,WIFE TURN,8,NC-17,223.69,155.001535,2
2,ZORRO ARK,8,NC-17,214.69,146.001535,3
3,GOODFELLAS SALUTE,8,PG,209.69,136.805246,4
4,SATURDAY LAMBS,8,G,204.72,136.508129,5
...,...,...,...,...,...,...
953,YOUNG LANGUAGE,2,G,6.93,-61.281871,953
954,STALLION SUNDANCE,2,PG-13,8.93,-62.709249,955
955,TEXAS WATCH,2,NC-17,5.94,-62.748465,956
956,FREEDOM CLEOPATRA,2,PG-13,5.95,-65.689249,957


---

## 8. Reflection on SQL Reasoning and Debugging (5 points)

Write **1–2 paragraphs** reflecting on your process. Address at least:
- How you broke down your mini-project into smaller SQL questions, or if your mini-project emerged/evolved as you explored the data and wrote individual queries.
- Any bugs, confusing results, or dead ends you hit and how you debugged them (e.g., row-count checks, `LIMIT`, CTE inspection, switching between Adminer and this notebook).
- Any tradeoffs you noticed between expressing logic with CTEs, `GROUP BY`, and window functions (e.g., readability vs. performance, ease of validation) either for this specific assignment or in general based on your experience.


> **_Write your reflection here._**
- Initially I tried to figure out queries that would show price and how popular certain movies were. I thought of grouping by genre but instead doing it with rating since it is a better proxy for people watching movies because historically only people above the age limit were allowed to rent certain movies (unless they had parent permission)
- I didn't have any bugs. Sometimes i had to add things to group me's just to get the query to run which did not make so much sense but would not run without. I just thought back to class when I had to add it in and it ended up working.
- I prefer using CTE's than using indented selects. One issue that I had was coming up with queries to make since most of the queries I thought of were complex. Most of the queries I have run have been quite efficient in my opinion. I don't really like window functions although they could be useful for ranking things like how i used it in this project. CTE's are much easier to debug than other complex queries.
- I had to revisit the EXPLAIN function ICA because I didn't know how to interperate the results. 
- I had some syntactical errors but i think that the readability of my queries made them easy enough to debug.

---

## 9. AI Audit Log (5 points)

This section is about **transparent, professional use of AI tools** (including systems like ChatGPT, GitHub Copilot, Perplexity, or others). You will not lose points for using AI, but you **must** document how you used it.

Answer the prompts below honestly. If you did **not** use any AI tools, you can say so explicitly.


### 9.1 Tools used

- List any AI tools you used while working on this assignment (e.g., ChatGPT, Copilot, Perplexity, Claude, etc.). If none, write "None".
- For each tool, briefly describe what you asked it to help with (e.g., "helped me remember `CREATE INDEX` syntax", "suggested a pattern for a window function").

> **_Describe your AI usage here._**
OpenAI:
<br>Query: 'potential questions to answer for movie business' - I used this to start thinking about potential questions I could answer
<br>Query: 'difference between RANK and DENSERANK' - I used this because I wanted to create a validation check and I needed to make sure that my check made sense.



### 9.2 Directly used vs. adapted

- Did you paste any AI-generated SQL or code directly into your notebook with minimal changes? If so, point to where (e.g., "Section 5 window query") and **make sure you add a comment in the code cell acknowledging the AI contribution**.
- For code or queries that you **adapted**, briefly describe what you changed and why (e.g., "adjusted table names to match Sakila, simplified to match our validation habits").

> **_Describe AI "copy-paste" usage here._**
<br>I didn't copy and paste any queries

### 9.3 Validation and trust

- How did you validate any AI-suggested queries or patterns before trusting them? (e.g., row-count checks, comparing against a simpler query, sanity checking outputs.)
- Are there any parts of your notebook where you are **not fully confident** in the result? If so, describe them, why you're lacking confidence, and what additional checks you would run if you had more time (e.g., "I wasn't sure if the window function was partitioned correctly, I would want to double-check the logic and maybe test it on a smaller dataset if I had more time").

> **_Describe your trust in AI outputs here._**
N/A

- - -

## 10. Creating your GitHub repo and pushing your code (10 points)

Now that you've finished your notebook, it's time to create a GitHub repository and push your code. 

**The goal for this section is to build a repository that someone could clone, spin up the Docker container to access the SQL database, and execute your notebook from top to bottom.**

To achieve this, follow these steps:

1. Create a new private repository on GitHub named `cmse492-hw02-yourMSUNetID` (replace `yourMSUNetID` with your actual MSU NetID).
2. Initialize the repository with a README file.
3. Clone the repository to your local machine.
4. Copy your completed notebook into the cloned repository folder. **Commit the fully-executed version so that someone could see the output _without_ having to run it, but using the Docker set up they should be able to execute it, if needed.**
5. Add the necessary Docker compose file needed to run the Sakila database (you can reuse the one provided in the course materials).
6. Update the README so that it provides an overview of the project, what you accomplished and the skills you're showcasing, and instructions on how to run the notebook and connect to the database (make sure you include information for starting up the Docker containers!).

    - You should be able to share this repository with a friend, co-worker, or future employer and they should be able to understand what you did, why you did it, and how to run your notebook and see your analysis in action.
    
7. Use Git commands to add, commit, and push your changes to GitHub. 

- - -
After you're made sure the GitHub repo is set up correctly, **Submit this notebook to D2L** and **Fill out the MS Form below** to finalize your submission.  
(If the form won't render in your notebook for some reason, you can also access it directly here: https://forms.office.com/r/Ppk0tnVkc8)

In [ ]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.office.com/r/Ppk0tnVkc8" 
	width="800" 
	height="800px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

- - -
### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework" section, find the appropriate submission link, and upload it there. **Make sure your instructors have access to your private GitHub repo by adding them as collaborators!**

See you in class!

---
&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.